# Coffee-17 Backbone Benchmark — tahan reset

Notebook ini menangani setup repo, GPU, Hugging Face Secrets, pemulihan dataset deterministik, restore checkpoint, training, evaluasi, dan sinkronisasi hasil.

Sebelum menjalankan:

1. Aktifkan GPU melalui **Runtime → Change runtime type → T4 GPU**.
2. Buat Hugging Face token dengan izin **write**.
3. Tambahkan token ke Colab Secrets dengan nama `HF_TOKEN` dan aktifkan akses notebook.
4. Jalankan semua cell dari atas. Setelah reset, jalankan ulang cell yang sama; checkpoint akan dipulihkan otomatis.


In [ ]:
# Konfigurasi yang boleh diubah. Cell utama tetap memiliki default jika cell ini terlewat.
SEEDS = [123]
BACKBONES = ['MV4', 'EV2', 'CV2', 'PV2']
HEADS = ['gap', 'hierarchical_gap', 'hbp']
HF_REPO_NAME = 'coffee-backbone-checkpoints'
HF_SYNC_EVERY = 2  # maksimum satu epoch perlu diulang setelah reset
EVALUATION_SPLIT = 'val'  # jangan ubah menjadi test saat screening


In [ ]:
# RUN EVERYTHING — cell mandiri dan aman dijalankan ulang.
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

seeds = list(globals().get('SEEDS', [123]))
backbones = list(globals().get('BACKBONES', ['MV4', 'EV2', 'CV2', 'PV2']))
heads = list(globals().get('HEADS', ['gap', 'hierarchical_gap', 'hbp']))
repo_name = str(globals().get('HF_REPO_NAME', 'coffee-backbone-checkpoints'))
sync_every = int(globals().get('HF_SYNC_EVERY', 2))
evaluation_split = str(globals().get('EVALUATION_SPLIT', 'val'))
assert evaluation_split == 'val', 'Screening backbone wajib memakai validation.'
assert sync_every > 0

if Path('/content').is_dir():
    platform = 'colab'
    workspace = Path('/content')
elif Path('/kaggle/working').is_dir():
    platform = 'kaggle'
    workspace = Path('/kaggle/working')
else:
    platform = 'local'
    workspace = Path.cwd()

repo = workspace / 'bilinear-LMMD'
repo_url = 'https://github.com/ediprin/bilinear-LMMD.git'
required_commit = '237160a'

def run(command, cwd=None):
    print('\n$', ' '.join(map(str, command)), flush=True)
    return subprocess.run([str(item) for item in command], cwd=cwd, check=True)

def safe_reset(path):
    resolved = path.resolve()
    root = workspace.resolve()
    assert resolved != root and resolved.is_relative_to(root), resolved
    if path.exists():
        print('Membersihkan state parsial:', path)
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()

print('=== 1/6 SETUP REPOSITORY ===')
if (repo / '.git').is_dir():
    run(['git', 'pull', '--ff-only'], cwd=repo)
else:
    if repo.exists():
        safe_reset(repo)
    run(['git', 'clone', repo_url, repo])

ancestor = subprocess.run(
    ['git', 'merge-base', '--is-ancestor', required_commit, 'HEAD'],
    cwd=repo,
).returncode
assert ancestor == 0, f'Repo belum memuat fitur artifact sync minimal {required_commit}.'
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)])

print('\n=== 2/6 GPU DAN HUGGING FACE ===')
import torch
assert torch.cuda.is_available(), 'GPU tidak tersedia. Aktifkan T4 GPU terlebih dahulu.'
print('GPU:', torch.cuda.get_device_name(0))

token = os.environ.get('HF_TOKEN')
if not token and platform == 'colab':
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError(
            'HF_TOKEN tidak dapat dibaca. Tambahkan di Colab Secrets dan aktifkan akses.'
        ) from exc
if not token and platform == 'kaggle':
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Tambahkan Kaggle Secret bernama HF_TOKEN.') from exc
assert token, 'HF_TOKEN kosong.'
os.environ['HF_TOKEN'] = token

from huggingface_hub import HfApi
api = HfApi(token=token)
identity = api.whoami()
hf_username = identity['name']
hf_repo = f'{hf_username}/{repo_name}'
api.create_repo(
    repo_id=hf_repo, repo_type='model', private=True, exist_ok=True, token=token
)
print('HF user:', hf_username)
print('HF repo:', hf_repo)

print('\n=== 3/6 PEMULIHAN DATASET ===')
dataset_base = workspace / 'coffee17-resumable-v1'
original = dataset_base / 'original'
clean = dataset_base / 'clean'
archive = dataset_base / 'coffee17.zip'
data_root = clean / 'folds/fold_1'
image_suffixes = {'.jpg', '.jpeg', '.png'}

def image_count(path):
    return sum(
        1 for item in path.rglob('*')
        if item.is_file() and item.suffix.lower() in image_suffixes
    ) if path.is_dir() else 0

original_valid = (original / 'source').is_dir() and image_count(original / 'source') == 979
if not original_valid:
    safe_reset(original)
    if archive.exists() and not zipfile.is_zipfile(archive):
        safe_reset(archive)
    run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17',
        '--output', original, '--archive', archive, '--seed', '42',
    ], cwd=repo)
assert image_count(original / 'source') == 979

audit_path = clean / 'audit.json'
clean_valid = False
if audit_path.is_file() and data_root.is_dir():
    try:
        audit = json.loads(audit_path.read_text())
        clean_valid = audit.get('status') == 'complete' and audit.get('clean_count') == 965
    except Exception:
        clean_valid = False
if not clean_valid:
    safe_reset(clean)
    run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds',
        '--source-root', original / 'source', '--output-root', clean,
        '--expected-count', '979', '--folds', '5', '--seed', '42',
        '--validation-ratio', '0.1',
    ], cwd=repo)
audit = json.loads(audit_path.read_text())
assert audit['status'] == 'complete' and audit['clean_count'] == 965
required_splits = [
    data_root / 'source/train', data_root / 'source/val', data_root / 'source/test'
]
assert all(path.is_dir() for path in required_splits)
print('Dataset:', data_root)
print('Input/Clean:', audit['input_count'], '/', audit['clean_count'])
print('Fold counts:', {path.name: image_count(path) for path in required_splits})

print('\n=== 4/6 STATUS CHECKPOINT REMOTE ===')
remote_files = api.list_repo_files(repo_id=hf_repo, repo_type='model', token=token)
remote_last = sorted(path for path in remote_files if path.endswith('/last.pt'))
print('Checkpoint last.pt remote:', len(remote_last))
for path in remote_last:
    print(' -', path)

print('\n=== 5/6 TRAIN / RESUME BACKBONE BENCHMARK ===')
output_root = workspace / 'backbone-results'
command = [
    sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_backbone_screening',
    '--data-root', str(data_root), '--output-root', str(output_root),
    '--seeds', *[str(seed) for seed in seeds],
    '--backbones', *backbones, '--heads', *heads,
    '--evaluation-split', evaluation_split,
    '--hf-repo', hf_repo, '--hf-sync-every', str(sync_every),
]
print('$', ' '.join(command), flush=True)
# Popen + heartbeat membuat status tetap terlihat walaupun tqdm child process
# tidak dirender oleh antarmuka notebook.
import time
model_codes = ['BV4G', 'BV4P', 'BV4H', 'BE2G', 'BE2P', 'BE2H', 'BC2G', 'BC2P', 'BC2H', 'BP2G', 'BP2P', 'BP2H']
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
next_heartbeat = started
while process.poll() is None:
    now = time.monotonic()
    if now >= next_heartbeat:
        progress_rows = []
        for code in model_codes:
            for seed in seeds:
                history_path = output_root / f'outputs/{code}_seed{seed}/history.json'
                if history_path.is_file():
                    try:
                        epochs_done = len(json.loads(history_path.read_text()))
                        progress_rows.append(f'{code}-s{seed}={epochs_done}/50')
                    except Exception:
                        progress_rows.append(f'{code}-s{seed}=syncing')
        elapsed_minutes = (now - started) / 60
        detail = ', '.join(progress_rows) if progress_rows else 'restore/download sedang berjalan'
        print(f'[HEARTBEAT {elapsed_minutes:.1f} menit] {detail}', flush=True)
        next_heartbeat = now + 60
    time.sleep(5)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'Benchmark berhenti dengan return code {return_code}. '
        'Penyebab asli tercetak tepat di atas pesan ini.'
    )

print('\n=== 6/6 SELESAI ===')
report_root = output_root / f'{evaluation_split}_reports'
leaderboard = report_root / 'backbone_leaderboard.json'
print('Output lokal:', output_root)
print('Backup HF  :', f'https://huggingface.co/{hf_repo}')
print('Leaderboard:', leaderboard)
assert leaderboard.is_file(), 'Leaderboard belum terbentuk.'


In [ ]:
# Tampilkan hasil tanpa bergantung pada variabel cell sebelumnya.
import json
from pathlib import Path

workspace = Path('/content') if Path('/content').is_dir() else Path('/kaggle/working')
leaderboard_path = workspace / 'backbone-results/val_reports/backbone_leaderboard.json'
assert leaderboard_path.is_file(), f'Belum ada: {leaderboard_path}'
rows = json.loads(leaderboard_path.read_text())
print('=== BACKBONE LEADERBOARD ===')
for rank, row in enumerate(rows, 1):
    print(
        f"{rank}. {row['code']:4s} {row['backbone_label']:27s} "
        f"{row['head'].upper():3s} Macro={row['macro_f1_mean']:.2%} "
        f"Hard={row['hard_class_f1_mean']:.2%} "
        f"Worst={row['worst_class_f1_mean']:.2%}"
    )


## Setelah runtime reset

Buka notebook ini lagi dan jalankan semua cell. Dataset direkonstruksi dengan seed dan audit yang sama. Runner memulihkan `last.pt`, `best.pt`, history, config, serta report dari repo Hugging Face, kemudian melanjutkan dari epoch terakhir yang tersinkronisasi. Jangan mengganti seed pembuatan dataset, jumlah fold, atau nama kelas ketika melanjutkan checkpoint lama.
